# Build 03-01 · Corrector inputs — targets ⨝ treatment, materialised per (version, split)

**Kernel: the analysis `.venv`** (`python3`). The one place the treatment join happens; the
corrector (03_02) and the re-evaluation (03_05) both read the artefact this notebook writes:

```
inputs/targets_<v>_<split>.parquet        (claim_id, date, observed)
   ⟕ (left join on claim_id) treatment    (decision [+ score])
   ──▶  mitigation/inputs/corrector_targets_<v>_<split>.parquet  (+ _meta.json)
```

Treatment source per version:
- **v3** — the v2 serving log's score file (`log_scores` kind, `logs/v2_score.parquet`):
  claim-level `score` + `decision` from the live model that generated v3's labels.
  Verified claim-unique on the company data (2026-09-02: `len == nunique`); a duplicate-event
  guard + collapse rule stays in §2 for reruns on refreshed exports.
- **v2** — the surviving **vehicle-status file** (§1 SOURCES). The real status vocabulary
  (user-described 2026-09-02): **fttl** (model fast-tracked → scrapped, label forced to 1),
  **repaired** / **total loss** (garage-verified outcomes), and three groups with **no usable
  outcome** — **awaiting authorisation** (repair decision still pending), **unrecovered**
  (stolen, never assessed: recorded total loss with neither garage verification nor an FTTL
  decision), and **NaN**. v2's original training swallowed all of them through the single
  observed column; **§4 drops the three unusable groups before the join**, so both splits and
  every downstream axis exclude them — the 03_02/03_03 **naive** axis thereby becomes the
  "v2 refit on known-outcome rows only" model. Verified to left-join onto the v2 training set
  with **zero unmatched rows** (2026-09). It carries `decision` only — the v1-era deciding
  **score is destroyed**, so v2's corrector_targets has no score column and rarity/pnu cannot
  run for v2 (thesis `tab:scheme-feasibility`); naive/transport can.

Built for **train + OOT** of each version, so 03_05 evaluates from the same artefact.

In [ ]:
# §0 — setup (analysis .venv kernel)
import json
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "src" / "config.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import config

pd.set_option("display.width", 160)
print("ROOT =", ROOT)

In [ ]:
# §1 — SOURCES / RUN_SPEC
ID_COL = "claim_id"
# Floor on the share of target rows the treatment source must cover. `matched.any()` is not a
# test: two UNRELATED integer key spaces still collide on a few rows and pass it — which is how
# v3's ID_CLAIM-keyed files joined "successfully" onto the claim-number-keyed v2 log
# (2026-09-03; since re-keyed by src/data/rekey_v3_claim_id.py). Lower it only with a reason
# printed next to it — e.g. a treatment source known to cover part of the window.
MIN_COVERAGE = 0.5
BUILD = {v: list(dict.fromkeys(["train", config.OOT_SPLIT[v]])) for v in ("v3", "v2")}

# v3 treatment: the v2 serving log's claim-level scores + decisions
V3_TREATMENT = config.path("log_scores", "v2")

# v2 treatment: the surviving vehicle-status file — company laptop only. Fill ALL of these in.
# NB if the file is a Z:-drive .pkl it is a JOBLIB dump (pd.read_pickle dies) — the reader below
# branches on the extension.
V2_STATUS_PATH = None                      # e.g. r"Z:/.../vehicle_status.pkl"; None -> skip v2
V2_STATUS_COLS = {"claim": "<<FILL IN>>",  # their claim-number column
                  "status": "<<FILL IN>>"} # their vehicle-status column

# The REAL status vocabulary (user-described 2026-09-02). Transcribe each value EXACTLY as it
# appears in the file — never guess a spelling; §4's audit fails loudly on any value declared
# in neither dict.
#
# KEPT — the only statuses with BOTH a treatment and a verified-or-forced outcome:
V2_STATUS_MAP = {
    # status value -> (decision, observed implied by the status)
    "<<fttl value>>":              (1, 1),  # model fast-tracked -> scrapped; label forced to 1
    "<<repaired value>>":          (0, 0),  # garage-verified repair (the negative class)
    "<<garage total loss value>>": (0, 1),  # garage-verified total loss
}
# DROPPED — no usable outcome, yet v2's original training swallowed them via the single
# observed column (status in {fttl, total loss, unrecovered} -> 1, else 0). NaN drops too (§4).
#   awaiting authorisation : repair-or-write-off decision still pending — outcome unknown
#   unrecovered            : stolen, never recovered — recorded total loss with neither a
#                            garage assessment nor an FTTL decision behind it
V2_STATUS_DROP = [
    "<<awaiting authorisation value>>",
    "<<unrecovered value>>",
]

print("build plan:", BUILD)
print("v3 treatment:", V3_TREATMENT)
print("v2 status   :", V2_STATUS_PATH or "(not set — v2 build will be skipped)")

In [ ]:
# §2 — helpers: the unique-claim gate, the collapse rule, the join+write step
def read_table(path) -> pd.DataFrame:
    """Parquet directly; a Z:-drive .pkl is a joblib dump, never pd.read_pickle."""
    p = str(path)
    if p.endswith(".parquet"):
        return pd.read_parquet(p)
    import joblib
    return joblib.load(p)


def norm_id(s: pd.Series) -> pd.Series:
    """claim_id as a stripped string, whatever dtype the source stored it in.

    The per-split exports and the log / vehicle-status sources do not agree on the claim_id
    dtype (int32 on one side, text on the other) and pandas refuses to merge across that.
    String is the safe common type: casting the other way would break a zero-padded or
    alphanumeric claim number. A float-typed read (any NaN in the column) renders as "12345.0",
    so it goes through Int64 first. This normalises the TYPE only — a padding or prefix
    difference still fails to match, which is what the coverage assertion in the join catches.
    """
    if pd.api.types.is_float_dtype(s):
        s = s.astype("Int64")
    return s.astype("string").str.strip()


def collapse_events(df: pd.DataFrame) -> pd.DataFrame:
    """One row per claim: a decision=1 event wins (its score); else the max-score event.

    Detection is the caller's unique-count gate; this runs ONLY when duplicates exist. Keeping
    a decision=0 event for a claim that was ever fast-tracked would misfile a scrapped car as
    garage-verified, and keeping the below-tau score of a scrapped claim would contradict the
    strict rule — sorting (decision desc, score desc) and keeping the first avoids both.
    """
    out = (df.sort_values(["decision", "score"], ascending=[False, False])
             .drop_duplicates(ID_COL, keep="first"))
    print(f"  collapsed {len(df):,} event rows -> {len(out):,} claims "
          f"({len(df) - len(out):,} duplicate events dropped)")
    return out


def build_corrector_targets(version: str, split: str, treatment: pd.DataFrame,
                            source_desc: str, observed_check: pd.DataFrame | None = None,
                            extra_meta: dict | None = None,
                            min_coverage: float = MIN_COVERAGE) -> Path:
    """targets(split) ⟕ treatment -> corrector_targets parquet + meta. Returns the path.

    Rows the treatment frame does not cover are dropped by the left join's notna gate — for v2
    that now includes the status-dropped groups (§4 filters them out of `treatment` first), so
    `n_unmatched_dropped` counts status drops and genuine non-coverage together; `extra_meta`
    carries the per-status breakdown that tells them apart.
    """
    t = pd.read_parquet(config.split_path("targets", version, split))
    id_dtype = t[ID_COL].dtype          # the artefact keeps the targets' own dtype (below)
    t[ID_COL] = norm_id(t[ID_COL])
    treatment = treatment.assign(**{ID_COL: norm_id(treatment[ID_COL])})
    if observed_check is not None:
        observed_check = observed_check.assign(**{ID_COL: norm_id(observed_check[ID_COL])})

    m = t.merge(treatment, on=ID_COL, how="left", validate="one_to_one")
    matched = m["decision"].notna()
    n_un = int((~matched).sum())
    cov = float(matched.mean())
    print(f"{version} {split}: {len(m):,} target rows | coverage {cov:.1%} "
          f"({n_un:,} unmatched -> dropped)")
    # Always show both id samples: a key-space mismatch is visible to the eye long before any
    # statistic — and the floor below is what makes it fatal rather than a low coverage number.
    print(f"  {ID_COL} e.g.  targets {t[ID_COL].head(3).tolist()}  |  "
          f"treatment {treatment[ID_COL].head(3).tolist()}")
    assert cov >= min_coverage, (
        f"{version} {split}: only {cov:.1%} of target rows matched the treatment on {ID_COL} "
        f"(floor {min_coverage:.0%}). The dtypes are aligned by norm_id, so the two sources key "
        f"differently — a row id vs the claim number (v3's ID_CLAIM, 2026-09-03), zero padding, "
        f"a prefix — compare the samples above and fix the SOURCE, never the join. If the "
        f"treatment source genuinely covers only part of this split, lower MIN_COVERAGE in §1 "
        f"with the reason written next to it.")

    n_mismatch = None
    if observed_check is not None:
        chk = m.merge(observed_check, on=ID_COL, how="left")
        both = chk["decision"].notna() & chk["observed_status"].notna()
        n_mismatch = int((chk.loc[both, "observed"].astype(int)
                          != chk.loc[both, "observed_status"].astype(int)).sum())
        if n_mismatch:
            print(f"  !! observed (targets) vs status-implied outcome disagree on {n_mismatch} rows")

    out = m.loc[matched].copy()
    out[ID_COL] = out[ID_COL].astype(id_dtype)   # normalisation was for the join only: every
    # downstream consumer joins this back onto features / score files, which still carry the
    # export dtype, so the written artefact must too (lossless — these rows came from targets).
    out["decision"] = out["decision"].astype(int)
    p = config.split_path("corrector_targets", version, split)
    p.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(p, index=False)
    p.with_name(p.stem + "_meta.json").write_text(json.dumps({
        "version": version, "split": split, "treatment_source": source_desc,
        "n_targets": int(len(m)), "n_unmatched_dropped": n_un,
        "n_out": int(len(out)), "has_score": bool("score" in out.columns),
        "n_scrapped": int(out["decision"].sum()),
        "n_observed_mismatch": n_mismatch,
        "columns": list(out.columns), "id_dtype": str(id_dtype),
        **(extra_meta or {}),
    }, indent=2), encoding="utf-8")
    print(f"  -> {p.name}  (cols: {list(out.columns)})")
    return p

In [ ]:
# §3 — v3: treatment from the v2 serving log (score + decision)
tr3 = read_table(V3_TREATMENT)
for c in (ID_COL, "score", "decision"):
    assert c in tr3.columns, f"log_scores is missing {c!r} (canonical names — re-run 01_export_v2_logs)"
tr3 = tr3[[ID_COL, "score", "decision"]].copy()
tr3[ID_COL] = norm_id(tr3[ID_COL])       # dtype-align before the gate (see norm_id in §2)
n, u = len(tr3), tr3[ID_COL].nunique()
print(f"v2 log_scores: {n:,} rows / {u:,} unique {ID_COL}")
if n != u:
    tr3 = collapse_events(tr3)      # no-op gate as of 2026-09-02 (n == u on the company data)

built = []
for sp in BUILD["v3"]:
    built.append(build_corrector_targets("v3", sp, tr3, str(V3_TREATMENT)))

In [ ]:
# §4 — v2: treatment from the vehicle-status file (decision only — no v1-era score)
if V2_STATUS_PATH is None:
    print("V2_STATUS_PATH not set — v2 build skipped (fill §1 SOURCES on the company laptop)")
else:
    st = read_table(V2_STATUS_PATH)
    claim_c, status_c = V2_STATUS_COLS["claim"], V2_STATUS_COLS["status"]
    for c in (claim_c, status_c):
        assert c in st.columns, f"status file has no column {c!r} — fix V2_STATUS_COLS"
    st = st[[claim_c, status_c]].rename(columns={claim_c: ID_COL}).copy()
    st[ID_COL] = norm_id(st[ID_COL])     # dtype-align before the gate (see norm_id in §2)

    # -- status audit: every value must be declared KEPT or DROPPED, nothing passes silently --
    counts = st[status_c].value_counts(dropna=False)
    print("status counts (full file):\n" + counts.to_string(), "\n")
    undeclared = sorted(set(counts.index.dropna()) - set(V2_STATUS_MAP) - set(V2_STATUS_DROP))
    assert not undeclared, (
        f"undeclared status values {undeclared[:10]} — add each to V2_STATUS_MAP (known outcome) "
        f"or V2_STATUS_DROP (no usable outcome) in §1; nothing is inferred")

    # -- the drop: no verified outcome -> excluded from training AND evaluation, here, once ----
    drop_mask = st[status_c].isna() | st[status_c].isin(V2_STATUS_DROP)
    drop_counts = {"<NaN>": int(st[status_c].isna().sum()),
                   **{s: int((st[status_c] == s).sum()) for s in V2_STATUS_DROP}}
    st = st.loc[~drop_mask].copy()
    print(f"dropped {int(drop_mask.sum()):,} rows with no usable outcome: {drop_counts}")

    st["decision"] = st[status_c].map({k: v[0] for k, v in V2_STATUS_MAP.items()}).astype(int)
    st["observed_status"] = st[status_c].map({k: v[1] for k, v in V2_STATUS_MAP.items()}).astype(int)
    assert (st["observed_status"] == 0).any(), (
        "no verified-negative (repaired) rows remain after the drop — the retrain would have no "
        "negative class. Either the repaired status value in V2_STATUS_MAP is misspelled, or the "
        "file genuinely has no repaired status; STOP and re-read the vocabulary off the file")

    n, u = len(st), st[ID_COL].nunique()
    kept_counts = {str(k): int(v) for k, v in st[status_c].value_counts().items()}
    print(f"vehicle-status kept: {n:,} rows / {u:,} unique {ID_COL} | {kept_counts}")
    assert n == u, "kept status rows are not claim-unique — decide a collapse rule before building"

    for sp in BUILD["v2"]:
        built.append(build_corrector_targets(
            "v2", sp, st[[ID_COL, "decision"]], str(V2_STATUS_PATH),
            observed_check=st[[ID_COL, "observed_status"]],
            extra_meta={"status_dropped": drop_counts, "status_kept": kept_counts}))

In [ ]:
# §5 — what exists now
rows = []
for v, sps in BUILD.items():
    for sp in sps:
        p = config.split_path("corrector_targets", v, sp)
        if p.is_file():
            meta = json.loads(p.with_name(p.stem + "_meta.json").read_text(encoding="utf-8"))
            rows.append({"version": v, "split": sp, "n_out": meta["n_out"],
                         "n_scrapped": meta["n_scrapped"], "has_score": meta["has_score"],
                         "unmatched_dropped": meta["n_unmatched_dropped"],
                         "observed_mismatch": meta["n_observed_mismatch"]})
        else:
            rows.append({"version": v, "split": sp, "n_out": "(not built)"})
display(pd.DataFrame(rows).set_index(["version", "split"]))

## Notes

- Unmatched rows (targets rows the treatment source never covers) are **dropped here, once** —
  every downstream consumer then works on the same treated population.
- **v2 status drop (2026-09-02)**: awaiting-authorisation, unrecovered, and NaN-status rows
  carry no verified outcome, so §4 removes them from the treatment frame before the join. The
  drop therefore reaches **train and OOT alike**: the naive/transport retrains (03_02 → 03_03)
  fit on known-outcome rows only, and 03_05 evaluates on the same filtered population. The
  per-status counts land in each `_meta.json` (`status_dropped` / `status_kept`).
- v2's files carry **no `score` column**; 03_02 detects that and skips rarity/pnu with a printed
  reason. v3's carry score + decision, so all four schemes run.
- Next: **03_02_reweight_mitigation.ipynb** (corrector), which now reads these files directly.